# Human Protein Atlas (HPA)

The **Human Protein Atlas** (HPA) is a Swedish-based programme that maps all human proteins in cells, tissues, and organs using integration of various omics technologies including antibody-based imaging, mass spectrometry-based proteomics, transcriptomics, and systems biology.

| Property | Detail |
|---|---|
| **Full name** | Human Protein Atlas |
| **URL** | https://www.proteinatlas.org |
| **API** | https://www.proteinatlas.org/api |
| **Bulk downloads** | https://www.proteinatlas.org/download/ |
| **ELIXIR CDR** | Yes — ELIXIR Core Data Resource |
| **Data types** | Protein expression (tissue, cell, blood, brain, pathology), subcellular localisation, antibody data |
| **Coverage** | ~20,000 human protein-coding genes |
| **License** | CC BY-SA 3.0 |
| **Key publications** | Uhlén et al., *Science* 2015; Thul et al., *Science* 2017 |

## Overview

HPA is divided into several atlases:

- **Tissue Atlas** — protein expression profiles across 44 normal human tissues
- **Cell Atlas** — subcellular localisation of proteins in human cell lines (confocal microscopy)
- **Pathology Atlas** — protein expression in cancers and correlation with patient survival
- **Blood Atlas** — proteins expressed in blood cells and their secretion into plasma
- **Brain Atlas** — regional and single-cell protein expression in the human brain
- **Single Cell Atlas** — single-cell RNA profiles across tissues

**Reference:** Uhlén M et al. (2015). Tissue-based map of the human proteome. *Science*, 347(6220). https://doi.org/10.1126/science.1260419

In [ ]:
import io
import time
import zipfile
from pathlib import Path

import polars as pl
import requests

## TODO

- [x] Ingest data
  - [x] Confirm API access
  - [x] Download bulk TSV (proteinatlas overview)
  - [x] Parse into Polars DataFrame
  - [x] Fetch single protein via API (TP53)
  - [x] Save to `data/` with caching
- [ ] Explore and clean data
- [ ] Analysis
- [ ] Visualization
- [ ] Statistical analysis

## 1. Ingest Data

### 1.1 Confirm API Access

The HPA JSON API returns per-gene data at:

```
https://www.proteinatlas.org/{GENE_NAME}.json
```

We first send a lightweight HEAD request to confirm connectivity.

In [ ]:
HPA_BASE_URL = "https://www.proteinatlas.org"
HPA_DOWNLOAD_URL = "https://www.proteinatlas.org/download"

# Probe the base URL with a HEAD request to confirm connectivity.
# Using a descriptive User-Agent is good practice for public APIs.
response = requests.head(
    HPA_BASE_URL,
    headers={"User-Agent": "elixir-of-life/1.0 (educational; contact: alice@example.com)"},
    timeout=10,
    allow_redirects=True,
)

print(f"Status : {response.status_code}")
print(f"Server : {response.headers.get('Server', 'unknown')}")
print(f"Access : {'OK' if response.status_code < 400 else 'FAILED'}")

### 1.2 Download Bulk TSV Data

HPA provides several pre-built bulk download files. We use the main overview file **`proteinatlas.tsv.zip`** which contains one row per gene with expression summary columns across all atlases.

Key columns include:
- `Gene` / `Gene synonym` — HGNC symbol and aliases  
- `Ensembl` — Ensembl gene ID  
- `Uniprot` — UniProt accession  
- `Chromosome`, `Position` — genomic coordinates  
- `Protein class` — functional classification  
- `Evidence` — evidence level (protein/transcript)  
- `HPA evidence`, `UniProt evidence`, `NeXtProt evidence` — per-source evidence  
- Tissue/cell/blood/brain expression level columns

In [ ]:
def download_hpa_bulk(filename: str, data_dir: Path, chunk_size: int = 1 << 20) -> Path:
    """Download a bulk HPA file with caching.

    Parameters
    ----------
    filename : str
        The filename on the HPA download server, e.g. ``"proteinatlas.tsv.zip"``.
    data_dir : Path
        Local directory in which to save the file.
    chunk_size : int, optional
        Download chunk size in bytes. Default is 1 MiB.

    Returns
    -------
    Path
        Absolute path to the saved file.

    Notes
    -----
    If the file already exists locally it is returned immediately without
    making a network request (cache-hit path).
    """
    data_dir.mkdir(parents=True, exist_ok=True)          # create data/ if absent
    dest = data_dir / filename

    if dest.exists():                                      # cache hit — skip download
        print(f"Cache hit: {dest} ({dest.stat().st_size / 1e6:.1f} MB)")
        return dest

    url = f"{HPA_DOWNLOAD_URL}/{filename}"
    print(f"Downloading {url} ...")
    t0 = time.perf_counter()

    headers = {"User-Agent": "elixir-of-life/1.0 (educational; contact: alice@example.com)"}
    with requests.get(url, headers=headers, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0))   # total bytes (may be 0)
        downloaded = 0
        with dest.open("wb") as fh:
            for chunk in r.iter_content(chunk_size=chunk_size):
                fh.write(chunk)
                downloaded += len(chunk)
                if total:
                    pct = downloaded / total * 100
                    print(f"\r  {pct:5.1f}%  ({downloaded / 1e6:.1f} / {total / 1e6:.1f} MB)", end="")

    elapsed = time.perf_counter() - t0
    print(f"\nDone in {elapsed:.1f}s — saved to {dest}")
    return dest


DATA_DIR = Path("data")
bulk_zip_path = download_hpa_bulk("proteinatlas.tsv.zip", DATA_DIR)

### 1.3 Parse into a Polars DataFrame

We unzip the downloaded archive in-memory and read the inner TSV directly into Polars.  
Polars `read_csv` with `separator="\t"` handles TSV natively; we use `infer_schema_length=0` first to inspect column names, then cast key columns to their appropriate dtypes.

In [ ]:
def parse_hpa_tsv_zip(zip_path: Path) -> pl.DataFrame:
    """Parse a zipped HPA TSV bulk download into a Polars DataFrame.

    Parameters
    ----------
    zip_path : Path
        Path to the ``.tsv.zip`` file downloaded from HPA.

    Returns
    -------
    pl.DataFrame
        Parsed DataFrame with corrected dtypes.  String columns encoding
        comma-separated lists (e.g. ``Gene synonym``, ``Uniprot``) are kept
        as ``Utf8``; numeric-looking columns are cast where unambiguous.

    Notes
    -----
    The zip archive contains exactly one TSV file.  We read it into a
    ``BytesIO`` buffer so no temporary files are written to disk.
    """
    with zipfile.ZipFile(zip_path) as zf:
        # The archive contains a single .tsv member
        tsv_name = next(n for n in zf.namelist() if n.endswith(".tsv"))
        print(f"Reading member: {tsv_name}")
        tsv_bytes = zf.read(tsv_name)

    # Read into Polars; infer_schema_length=10000 gives schema inference a
    # generous window given the wide variety of expression level strings.
    df = pl.read_csv(
        io.BytesIO(tsv_bytes),
        separator="\t",
        infer_schema_length=10_000,
        null_values=["", "NA", "N/A", "-"],
        ignore_errors=True,
    )
    return df


df_hpa = parse_hpa_tsv_zip(bulk_zip_path)

print(f"Shape : {df_hpa.shape[0]:,} rows × {df_hpa.shape[1]} columns")
print("\nSchema (first 20 columns):")
for col, dtype in list(df_hpa.schema.items())[:20]:
    print(f"  {col:<45} {dtype}")

In [ ]:
# Quick preview: first 5 rows, first 8 columns
df_hpa.select(df_hpa.columns[:8]).head(5)

### 1.4 Fetch a Specific Protein via the HPA JSON API

The HPA provides a per-entry JSON API at:

```
https://www.proteinatlas.org/{GENE}.json
```

This returns structured data for a single gene/protein, including tissue expression,
antibody information, subcellular localisation, and links to images.
We demonstrate this for **TP53** (tumour suppressor p53), one of the most studied human proteins.

In [ ]:
def fetch_protein_json(gene: str, data_dir: Path, delay: float = 1.0) -> dict:
    """Fetch and cache the HPA JSON record for a single gene.

    Parameters
    ----------
    gene : str
        HGNC gene symbol, e.g. ``"TP53"``.
    data_dir : Path
        Directory for caching JSON responses.
    delay : float, optional
        Polite delay in seconds before making a live request. Default 1.0 s.

    Returns
    -------
    dict
        Parsed JSON response from the HPA API.

    Raises
    ------
    requests.HTTPError
        If the server returns a non-2xx status code.

    Notes
    -----
    Responses are cached as ``{data_dir}/{gene}.json`` so repeated calls
    do not hit the server.  The HPA terms of service request polite crawling;
    the ``delay`` parameter enforces this.
    """
    data_dir.mkdir(parents=True, exist_ok=True)
    cache_file = data_dir / f"{gene}.json"

    if cache_file.exists():                               # cache hit
        import json
        print(f"Cache hit: {cache_file}")
        with cache_file.open() as fh:
            return json.load(fh)

    time.sleep(delay)                                     # polite delay before request
    url = f"{HPA_BASE_URL}/{gene}.json"
    print(f"Fetching {url} ...")

    headers = {"User-Agent": "elixir-of-life/1.0 (educational; contact: alice@example.com)"}
    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()

    data = r.json()

    import json
    with cache_file.open("w") as fh:                     # persist to disk
        json.dump(data, fh, indent=2)
    print(f"Saved to {cache_file}")
    return data


tp53_data = fetch_protein_json("TP53", DATA_DIR)

# Top-level keys reveal the structure of the response
print("Top-level keys:", list(tp53_data.keys())[:15])